# 01 - EDA + CBF + CARS TourHub Bali

Notebook ini dibuat sebagai langkah awal sebelum FastAPI. Tujuannya: memahami dataset, membersihkan data, lalu mencoba rekomendasi sederhana dengan Content-Based Filtering dan Context-Aware Recommender System.


In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


## 1. Load dataset


In [2]:
DATA_PATH = Path('../data/bali_tourist_destination.csv')
df = pd.read_csv(DATA_PATH)
df.head()


,id_tempat,nama_tempat_wisata,kategori,kecamatan,kabupaten_kota,rating,jumlah_rating,latitude,longitude,link_google_maps,link_gambar
0,BL0101001,Patung Titi Banda,Umum,Denpasar Barat,Kota Denpasar,4.6,1941,-8.649292,115.255043,https://www.google.com/maps/place/Patung+Titi+...,"https://loremflickr.com/1080/720/bali,statue?l..."
1,BL0101002,Uma.palak.(parkir.2),Rekreasi,Denpasar Barat,Kota Denpasar,4.8,83,-8.617877,115.212975,https://www.google.com/maps/place/Uma.palak.%2...,"https://loremflickr.com/1080/720/bali,garden?l..."
2,BL0101003,Tukad Bindu Park,Rekreasi,Denpasar Barat,Kota Denpasar,4.5,1180,-8.643791,115.235812,https://www.google.com/maps/place/Tukad+Bindu+...,"https://loremflickr.com/1080/720/bali,garden?l..."
3,BL0101004,Pantai Padang Galak,Alam,Denpasar Barat,Kota Denpasar,4.6,486,-8.661138,115.263301,https://www.google.com/maps/place/Pantai+Padan...,"https://loremflickr.com/1080/720/bali,beach?lo..."
4,BL0101005,Museum Le Mayeur,Budaya,Denpasar Barat,Kota Denpasar,4.1,679,-8.674877,115.263678,https://www.google.com/maps/place/Museum+Le+Ma...,"https://loremflickr.com/1080/720/bali,museum?l..."


In [3]:
print('Shape:', df.shape)
display(df.info())
display(df.describe(include='all'))


Shape: (1452, 11)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1452 entries, 0 to 1451
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_tempat           1452 non-null   object 
 1   nama_tempat_wisata  1452 non-null   object 
 2   kategori            1452 non-null   object 
 3   kecamatan           1452 non-null   object 
 4   kabupaten_kota      1452 non-null   object 
 5   rating              1452 non-null   float64
 6   jumlah_rating       1452 non-null   int64  
 7   latitude            1452 non-null   float64
 8   longitude           1452 non-null   float64
 9   link_google_maps    1452 non-null   object 
 10  link_gambar         1452 non-null   object 
dtypes: float64(3), int64(1), object(7)
memory usage: 124.9+ KB


None

,id_tempat,nama_tempat_wisata,kategori,kecamatan,kabupaten_kota,rating,jumlah_rating,latitude,longitude,link_google_maps,link_gambar
count,1452,1452,1452,1452,1452,1452.000000,1452.000000,1452.000000,1452.000000,1452,1452
unique,1452,1437,4,56,9,NaN,NaN,NaN,NaN,1452,1452
top,BL0101001,Pantai Pandawa,Umum,Abiansemal,Kabupaten Badung,NaN,NaN,NaN,NaN,https://www.google.com/maps/place/Patung+Titi+...,"https://loremflickr.com/1080/720/bali,statue?l..."
freq,1,2,577,71,261,NaN,NaN,NaN,NaN,1,1
mean,NaN,NaN,NaN,NaN,NaN,4.505028,1176.176997,-8.459082,115.204685,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN,0.561069,5268.544496,0.182956,0.322467,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,-8.849991,111.833469,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,4.400000,24.000000,-8.579316,115.139314,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,4.600000,110.000000,-8.465729,115.236178,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,4.800000,472.000000,-8.328480,115.364814,NaN,NaN


## 2. Cek kategori dan lokasi


In [4]:
display(df['kategori'].value_counts())
display(df['kabupaten_kota'].value_counts())


kategori
Umum        577
Alam        529
Budaya      182
Rekreasi    164
Name: count, dtype: int64

kabupaten_kota
Kabupaten Badung        261
Kabupaten Gianyar       215
Kabupaten Tabanan       170
Kabupaten Buleleng      169
Kabupaten Karangasem    158
Kabupaten Klungkung     136
Kota Denpasar           130
Kabupaten Bangli        110
Kabupaten Jembrana      103
Name: count, dtype: int64

## 3. Preprocessing sederhana


In [5]:
def normalize_text(value):
    if pd.isna(value):
        return ''
    text = str(value).lower().strip()
    text = re.sub(r'[^a-z0-9A-ZÀ-ÿ\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def infer_tipe_wisata(row):
    name = normalize_text(row['nama_tempat_wisata'])
    kategori = normalize_text(row['kategori'])
    indoor_keywords = ['museum', 'galeri', 'gallery', 'mall', 'plaza', 'theater', 'teater', 'studio']
    outdoor_keywords = ['pantai', 'beach', 'air terjun', 'waterfall', 'bukit', 'hill', 'gunung', 'danau', 'lake', 'taman', 'park', 'sawah', 'river', 'rafting', 'snorkeling', 'diving', 'zoo', 'monkey forest', 'forest', 'pura', 'temple', 'goa', 'cave']
    if any(k in name for k in indoor_keywords):
        return 'indoor'
    if any(k in name for k in outdoor_keywords):
        return 'outdoor'
    if kategori == 'alam':
        return 'outdoor'
    if kategori == 'budaya':
        return 'mixed'
    return 'mixed'

clean = df.copy()
for col in ['id_tempat', 'nama_tempat_wisata', 'kategori', 'kecamatan', 'kabupaten_kota']:
    clean[col] = clean[col].fillna('').astype(str).str.strip()

clean['rating'] = pd.to_numeric(clean['rating'], errors='coerce').fillna(0)
clean['jumlah_rating'] = pd.to_numeric(clean['jumlah_rating'], errors='coerce').fillna(0).astype(int)
clean = clean.drop_duplicates(subset=['nama_tempat_wisata', 'latitude', 'longitude'])
clean['tipe_wisata'] = clean.apply(infer_tipe_wisata, axis=1)
clean['rating_score'] = (clean['rating'] / clean['rating'].max()).clip(0, 1)
log_reviews = np.log1p(clean['jumlah_rating'])
clean['popularity_score'] = (log_reviews / log_reviews.max()).clip(0, 1)
clean['feature_text'] = (
    clean['nama_tempat_wisata'].map(normalize_text) + ' ' +
    clean['kategori'].map(normalize_text) + ' ' +
    clean['kecamatan'].map(normalize_text) + ' ' +
    clean['kabupaten_kota'].map(normalize_text) + ' ' +
    clean['tipe_wisata'].map(normalize_text)
)
clean[['nama_tempat_wisata','kategori','kabupaten_kota','rating','jumlah_rating','tipe_wisata','feature_text']].head()


,nama_tempat_wisata,kategori,kabupaten_kota,rating,jumlah_rating,tipe_wisata,feature_text
0,Patung Titi Banda,Umum,Kota Denpasar,4.6,1941,mixed,patung titi banda umum denpasar barat kota den...
1,Uma.palak.(parkir.2),Rekreasi,Kota Denpasar,4.8,83,outdoor,uma palak parkir 2 rekreasi denpasar barat kot...
2,Tukad Bindu Park,Rekreasi,Kota Denpasar,4.5,1180,outdoor,tukad bindu park rekreasi denpasar barat kota ...
3,Pantai Padang Galak,Alam,Kota Denpasar,4.6,486,outdoor,pantai padang galak alam denpasar barat kota d...
4,Museum Le Mayeur,Budaya,Kota Denpasar,4.1,679,indoor,museum le mayeur budaya denpasar barat kota de...


## 4. Content-Based Filtering (CBF)


In [6]:
vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
destination_matrix = vectorizer.fit_transform(clean['feature_text'])
destination_matrix.shape


(1447, 5630)

## 5. CARS: context scoring sederhana


In [7]:
def weather_group(weather):
    text = normalize_text(weather)
    if any(x in text for x in ['hujan', 'rain', 'shower', 'storm', 'petir', 'gerimis']):
        return 'hujan'
    if any(x in text for x in ['cerah', 'clear', 'sunny']):
        return 'cerah'
    if any(x in text for x in ['berawan', 'cloud', 'mendung', 'overcast']):
        return 'berawan'
    return 'tidak_diketahui'

def context_multiplier(tipe_wisata, popularity_score, weather=None, visit_day=None, is_high_season=False):
    multiplier = 1.0
    group = weather_group(weather)
    tipe = str(tipe_wisata).lower()

    if group == 'hujan':
        if tipe == 'outdoor':
            multiplier *= 0.72
        elif tipe == 'indoor':
            multiplier *= 1.12
        else:
            multiplier *= 0.92
    elif group == 'cerah':
        if tipe == 'outdoor':
            multiplier *= 1.08
        elif tipe == 'indoor':
            multiplier *= 0.96

    if normalize_text(visit_day) in ['weekend', 'akhir pekan', 'sabtu', 'minggu']:
        multiplier *= 0.90 if popularity_score >= 0.75 else 1.03

    if is_high_season:
        multiplier *= 0.88 if popularity_score >= 0.75 else 1.05

    return multiplier


## 6. Fungsi rekomendasi final


In [8]:
def recommend_destinations(kategori_preferensi=None, kabupaten_kota=None, kecamatan=None, keywords=None, min_rating=None, top_n=10, weather=None, visit_day=None, is_high_season=False):
    kategori_preferensi = kategori_preferensi or []
    keywords = keywords or []
    user_query = ' '.join([
        *[normalize_text(x) for x in kategori_preferensi],
        normalize_text(kabupaten_kota),
        normalize_text(kecamatan),
        *[normalize_text(x) for x in keywords],
    ]).strip() or 'wisata bali'

    user_vector = vectorizer.transform([user_query])
    cbf_scores = cosine_similarity(user_vector, destination_matrix).flatten()

    result = clean.copy()
    result['cbf_score'] = cbf_scores

    if kabupaten_kota:
        result = result[result['kabupaten_kota'].str.lower() == str(kabupaten_kota).lower()]
    if kecamatan:
        result = result[result['kecamatan'].str.lower() == str(kecamatan).lower()]
    if min_rating is not None:
        result = result[result['rating'] >= min_rating]

    result['context_multiplier'] = result.apply(
        lambda row: context_multiplier(row['tipe_wisata'], row['popularity_score'], weather, visit_day, is_high_season),
        axis=1
    )

    result['base_score'] = (0.70 * result['cbf_score']) + (0.20 * result['rating_score']) + (0.10 * result['popularity_score'])
    result['final_score'] = result['base_score'] * result['context_multiplier']

    cols = ['nama_tempat_wisata', 'kategori', 'tipe_wisata', 'kecamatan', 'kabupaten_kota', 'rating', 'jumlah_rating', 'cbf_score', 'context_multiplier', 'final_score']
    return result.sort_values('final_score', ascending=False)[cols].head(top_n)


## 7. Contoh rekomendasi


In [9]:
recommend_destinations(
    kategori_preferensi=['Alam', 'Budaya'],
    kabupaten_kota='Kabupaten Gianyar',
    keywords=['pantai', 'sunset'],
    min_rating=4.2,
    top_n=10,
    weather='hujan',
    visit_day='weekend',
    is_high_season=False
)


,nama_tempat_wisata,kategori,tipe_wisata,kecamatan,kabupaten_kota,rating,jumlah_rating,cbf_score,context_multiplier,final_score
475,Museum Puri Lukisan,Budaya,indoor,Gianyar,Kabupaten Gianyar,4.3,2195,0.195610,1.1536,0.433376
461,The Blanco Renaissance Museum,Budaya,indoor,Gianyar,Kabupaten Gianyar,4.3,2097,0.175004,1.1536,0.416279
593,Museum Seni Agung Rai,Budaya,indoor,Ubud,Kabupaten Gianyar,4.5,1464,0.134704,1.1536,0.389371
587,Ada Garuda Museum,Budaya,indoor,Tegallalang,Kabupaten Gianyar,4.7,209,0.146003,1.1536,0.388285
458,Istana Kepresidenan Tampaksiring Bali,Budaya,mixed,Gianyar,Kabupaten Gianyar,4.7,1544,0.192581,0.9476,0.366249
466,Patung Bayi,Umum,mixed,Gianyar,Kabupaten Gianyar,4.6,2408,0.172155,0.9476,0.352561
482,Alun-Alun Kota Gianyar,Umum,mixed,Gianyar,Kabupaten Gianyar,4.5,3279,0.172712,0.9476,0.351677
474,Bangkiang Jaran,Umum,mixed,Gianyar,Kabupaten Gianyar,4.9,365,0.168158,0.9476,0.345791
449,Seraya Budaya,Umum,mixed,Blahbatuh,Kabupaten Gianyar,4.8,77,0.192524,0.9476,0.345456
469,"Delodsema Traditional Village, Tegallalang",Budaya,mixed,Gianyar,Kabupaten Gianyar,4.6,146,0.185283,0.9476,0.338282


## 8. Simpan dataset bersih

Dataset bersih ini bisa dipakai sebagai pembanding atau dimasukkan ke database nanti.


In [10]:
OUTPUT_PATH = Path('../data/cleaned_bali_tourist_destination.csv')
clean.to_csv(OUTPUT_PATH, index=False)
OUTPUT_PATH


PosixPath('../data/cleaned_bali_tourist_destination.csv')